# 🛠️ Quy Trình Tiền Xử Lý Dữ Liệu & Gộp Tập Dữ Liệu YOLO Pothole Detection

Notebook này thực hiện hai nhiệm vụ chính:
1. **Phần 1: Tiền xử lý tập dữ liệu RDD-2022**:
   - Loại bỏ hoàn toàn các ảnh thuộc nguồn `China_Drone`.
   - Chỉ lọc giữ lại nhãn ổ gà (Lớp 4) và chuyển đổi chỉ số lớp này thành lớp `0`.
   - Lấy chính xác **7%** lượng ảnh nền (ảnh không chứa ổ gà sau khi đã loại China_Drone) để giúp mô hình có tính khái quát hóa cao hơn.
   - Gộp tập Train và tập Test của RDD-2022 thành tập Train chung, giữ riêng tập Val.
2. **Phần 2: Gộp dữ liệu liên tập (Cross Dataset)**:
   - Gộp tập Train/Test đã xử lý của RDD-2022 với tập Train/Test của BharatPothole thành tập **Train chung**.
   - Gộp tập Val đã xử lý của RDD-2022 với tập Valid của BharatPothole thành tập **Val chung**.
   - Thống kê chi tiết số liệu của tập dữ liệu gộp cuối cùng.

## ⚙️ Khởi Tạo và Cấu Hình Thư Mục

In [3]:
import os
import random
import shutil
from pathlib import Path
from tqdm.notebook import tqdm
import pandas as pd
import IPython.display as display

PROJECT_ROOT = Path(r'd:\Research\Yolo_Pothole_detection\Pothole_Detection')
RAW_RDD_DIR = PROJECT_ROOT / 'data' / 'raw' / 'rdd2022' / 'RDD_SPLIT'
RAW_BHARAT_DIR = PROJECT_ROOT / 'data' / 'raw' / 'bharatpothole' / 'BharatPotHole' / 'BharatPotHole'

PROCESSED_RDD_DIR = PROJECT_ROOT / 'data' / 'processed' / 'rdd2022_processed'
COMBINED_DIR = PROJECT_ROOT / 'data' / 'processed' / 'combined_pothole'

# Tạo các thư mục đầu ra nếu chưa tồn tại
PROCESSED_RDD_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Khởi tạo cấu hình và thư mục thành công.")

✅ Khởi tạo cấu hình và thư mục thành công.


## 1️⃣ Phần 1: Tiền Xử Lý Tập Dữ Liệu RDD-2022

In [4]:
def process_rdd_split(split_name, src_root, dest_split_dir):
    """
    Tiền xử lý dữ liệu RDD-2022:
    - Lọc bỏ ảnh China_Drone
    - Lọc giữ lại nhãn lớp 4 (Pothole) và chuyển thành lớp 0
    - Lấy ngẫu nhiên 7% ảnh nền (background)
    """
    src_img_dir = src_root / split_name / 'images'
    src_lbl_dir = src_root / split_name / 'labels'
    
    dest_img_dir = dest_split_dir / 'images'
    dest_lbl_dir = dest_split_dir / 'labels'
    
    dest_img_dir.mkdir(parents=True, exist_ok=True)
    dest_lbl_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Tìm tất cả ảnh và loại bỏ China_Drone
    img_extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
    all_images = []
    for ext in img_extensions:
        all_images.extend(src_img_dir.glob(f'*{ext}'))
    
    filtered_images = [img for img in all_images if "China_Drone" not in img.name]
    
    pothole_images = []
    background_images = []
    
    # 2. Phân loại ảnh có chứa ổ gà và ảnh nền
    for img_path in filtered_images:
        lbl_path = src_lbl_dir / f"{img_path.stem}.txt"
        has_pothole = False
        
        if lbl_path.exists():
            with open(lbl_path, 'r') as f:
                lines = [line.strip() for line in f if line.strip()]
                for line in lines:
                    parts = line.split()
                    if len(parts) >= 5 and int(parts[0]) == 4:  # ID 4 là Pothole
                        has_pothole = True
                        break
                        
        if has_pothole:
            pothole_images.append(img_path)
        else:
            background_images.append(img_path)
            
    # 3. Lấy ngẫu nhiên 7% ảnh nền
    random.seed(42)
    num_bg_keep = int(len(background_images) * 0.07)
    selected_bg = random.sample(background_images, num_bg_keep) if num_bg_keep > 0 else []
    
    # 4. Sao chép hình ảnh và tạo file nhãn tương ứng
    final_images = pothole_images + selected_bg
    print(f"\n⚙️ Đang xử lý RDD-2022 split [{split_name}] -> [{dest_split_dir.name}]:")
    print(f"   • Số ảnh ban đầu (đã loại China_Drone): {len(filtered_images):,}")
    print(f"   • Số ảnh Pothole (giữ lại 100%): {len(pothole_images):,}")
    print(f"   • Số ảnh nền gốc: {len(background_images):,} -> Giữ lại 7%: {len(selected_bg):,}")
    print(f"   • Tổng số ảnh lưu sang thư mục đích: {len(final_images):,}")
    
    for img_path in tqdm(final_images, desc=f"Đang copy {split_name}"):
        # Sao chép ảnh
        shutil.copy2(img_path, dest_img_dir / img_path.name)
        
        # Xử lý nhãn
        dest_lbl_path = dest_lbl_dir / f"{img_path.stem}.txt"
        lbl_path = src_lbl_dir / f"{img_path.stem}.txt"
        
        if img_path in pothole_images and lbl_path.exists():
            new_lines = []
            with open(lbl_path, 'r') as f:
                lines = [line.strip() for line in f if line.strip()]
                for line in lines:
                    parts = line.split()
                    if len(parts) >= 5 and int(parts[0]) == 4:
                        parts[0] = '0'  # Chuyển đổi ID lớp 4 thành lớp 0
                        new_lines.append(" ".join(parts))
            with open(dest_lbl_path, 'w') as f:
                f.write("\n".join(new_lines) + "\n")
        else:
            # Tạo file nhãn trống cho ảnh nền
            with open(dest_lbl_path, 'w') as f:
                pass

In [5]:
# Tiến hành tiền xử lý và gộp các tập dữ liệu RDD-2022
# 1. Gộp RDD Train -> PROCESSED_RDD_DIR / 'train'
process_rdd_split('train', RAW_RDD_DIR, PROCESSED_RDD_DIR / 'train')

# 2. Gộp RDD Test -> PROCESSED_RDD_DIR / 'train' (Gộp Train và Test)
process_rdd_split('test', RAW_RDD_DIR, PROCESSED_RDD_DIR / 'train')

# 3. RDD Val -> PROCESSED_RDD_DIR / 'val' (Giữ riêng)
process_rdd_split('val', RAW_RDD_DIR, PROCESSED_RDD_DIR / 'val')


⚙️ Đang xử lý RDD-2022 split [train] -> [train]:
   • Số ảnh ban đầu (đã loại China_Drone): 50,388
   • Số ảnh Pothole (giữ lại 100%): 5,104
   • Số ảnh nền gốc: 45,284 -> Giữ lại 7%: 3,169
   • Tổng số ảnh lưu sang thư mục đích: 8,273


Đang copy train:   0%|          | 0/8273 [00:00<?, ?it/s]


⚙️ Đang xử lý RDD-2022 split [test] -> [train]:
   • Số ảnh ban đầu (đã loại China_Drone): 10,762
   • Số ảnh Pothole (giữ lại 100%): 1,046
   • Số ảnh nền gốc: 9,716 -> Giữ lại 7%: 680
   • Tổng số ảnh lưu sang thư mục đích: 1,726


Đang copy test:   0%|          | 0/1726 [00:00<?, ?it/s]


⚙️ Đang xử lý RDD-2022 split [val] -> [val]:
   • Số ảnh ban đầu (đã loại China_Drone): 10,818
   • Số ảnh Pothole (giữ lại 100%): 1,070
   • Số ảnh nền gốc: 9,748 -> Giữ lại 7%: 682
   • Tổng số ảnh lưu sang thư mục đích: 1,752


Đang copy val:   0%|          | 0/1752 [00:00<?, ?it/s]

## 2️⃣ Phần 2: Gộp Dữ Liệu Liên Tập (Cross Dataset - `combined_pothole`)

Chúng ta sẽ thực hiện gộp dữ liệu từ `rdd2022_processed` và `BharatPotHole` thành tập dữ liệu gộp cuối cùng:

In [6]:
def merge_dataset_split(src_dirs, dest_split_dir):
    """
    Gộp nhiều thư mục nguồn (chứa images/ và labels/) vào một thư mục đích chung
    """
    dest_images = dest_split_dir / 'images'
    dest_labels = dest_split_dir / 'labels'
    
    dest_images.mkdir(parents=True, exist_ok=True)
    dest_labels.mkdir(parents=True, exist_ok=True)
    
    total_images_copied = 0
    total_labels_copied = 0
    
    for src in src_dirs:
        src_images = src / 'images'
        src_labels = src / 'labels'
        
        if not src_images.exists() or not src_labels.exists():
            print(f"⚠️ Cảnh báo: Thư mục nguồn {src} không tồn tại thư mục images hoặc labels!")
            continue
            
        # Sao chép ảnh
        img_files = list(src_images.glob('*'))
        for f in tqdm(img_files, desc=f"Gộp ảnh từ {src.parent.name}/{src.name}", leave=False):
            if f.is_file():
                shutil.copy2(f, dest_images / f.name)
                total_images_copied += 1
                
        # Sao chép nhãn
        lbl_files = list(src_labels.glob('*'))
        for f in tqdm(lbl_files, desc=f"Gộp nhãn từ {src.parent.name}/{src.name}", leave=False):
            if f.is_file():
                shutil.copy2(f, dest_labels / f.name)
                total_labels_copied += 1
                
    print(f"✅ Đã gộp thành công vào [{dest_split_dir.name}]: {total_images_copied:,} ảnh và {total_labels_copied:,} file nhãn.")

In [7]:
# 1. Gộp tập Train chung:
# - RDD-2022 processed train (gồm train + test)
# - BharatPothole train
# - BharatPothole test
print("🔄 Bắt đầu gộp tập TRAIN chung...")
merge_dataset_split([
    PROCESSED_RDD_DIR / 'train',
    RAW_BHARAT_DIR / 'train',
    RAW_BHARAT_DIR / 'test'
], COMBINED_DIR / 'train')

# 2. Gộp tập Val chung:
# - RDD-2022 processed val
# - BharatPothole valid
print("\n🔄 Bắt đầu gộp tập VAL chung...")
merge_dataset_split([
    PROCESSED_RDD_DIR / 'val',
    RAW_BHARAT_DIR / 'valid'
], COMBINED_DIR / 'val')

🔄 Bắt đầu gộp tập TRAIN chung...


Gộp ảnh từ rdd2022_processed/train:   0%|          | 0/6810 [00:00<?, ?it/s]

Gộp nhãn từ rdd2022_processed/train:   0%|          | 0/6810 [00:00<?, ?it/s]

Gộp ảnh từ BharatPotHole/train:   0%|          | 0/5067 [00:00<?, ?it/s]

Gộp nhãn từ BharatPotHole/train:   0%|          | 0/5067 [00:00<?, ?it/s]

...

Gộp ảnh từ rdd2022_processed/val:   0%|          | 0/1186 [00:00<?, ?it/s]

Gộp nhãn từ rdd2022_processed/val:   0%|          | 0/1186 [00:00<?, ?it/s]

Gộp ảnh từ BharatPotHole/valid:   0%|          | 0/1345 [00:00<?, ?it/s]

Gộp nhãn từ BharatPotHole/valid:   0%|          | 0/1345 [00:00<?, ?it/s]

✅ Đã gộp thành công vào [val]: 2,531 ảnh và 2,531 file nhãn.


## 📊 Thống Kê Số Liệu Tập Dữ Liệu Gộp Cuối Cùng

In [8]:
def analyze_combined_dataset(dataset_dir):
    stats = {}
    splits = ['train', 'val']
    
    for split in splits:
        img_dir = dataset_dir / split / 'images'
        lbl_dir = dataset_dir / split / 'labels'
        
        all_images = list(img_dir.glob('*'))
        all_labels = list(lbl_dir.glob('*.txt'))
        
        total_images = len(all_images)
        
        pothole_boxes = 0
        images_with_pothole = 0
        
        for lbl_path in all_labels:
            has_pothole = False
            if lbl_path.exists():
                with open(lbl_path, 'r') as f:
                    lines = [line.strip() for line in f if line.strip()]
                    for line in lines:
                        parts = line.split()
                        if len(parts) >= 5 and int(parts[0]) == 0:  # Lớp 0 hiện tại là Pothole
                            pothole_boxes += 1
                            has_pothole = True
            
            if has_pothole:
                images_with_pothole += 1
                
        real_background_images = total_images - images_with_pothole
        
        stats[split] = {
            'Tổng số ảnh': total_images,
            'Số ảnh chứa Ổ gà (Lớp 0)': images_with_pothole,
            'Số ảnh nền (Background)': real_background_images,
            'Tỷ lệ ảnh nền (%)': f"{round(real_background_images / total_images * 100, 2)}%" if total_images > 0 else "0%",
            'Tổng số nhãn Ổ gà (BBox)': pothole_boxes
        }
        
    df_stats = pd.DataFrame.from_dict(stats, orient='index')
    return df_stats

print("📊 BẢNG THỐNG KÊ CHI TIẾT TẬP DỮ LIỆU GỘP CUỐI CÙNG:")
df_result = analyze_combined_dataset(COMBINED_DIR)
display.display(df_result)

📊 BẢNG THỐNG KÊ CHI TIẾT TẬP DỮ LIỆU GỘP CUỐI CÙNG:


,Tổng số ảnh,Số ảnh chứa Ổ gà (Lớp 0),Số ảnh nền (Background),Tỷ lệ ảnh nền (%),Tổng số nhãn Ổ gà (BBox)
train,12539,6515,6024,48.04%,15452
val,2531,1291,1240,48.99%,3227
